In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from darts import TimeSeries

from darts.utils.missing_values import missing_values_ratio, fill_missing_values
from darts.metrics.metrics import mape, mae, rmse, mase

from ts_missing_values.gap_filling import *
from ts_missing_values.comparison import *
from ts_missing_values.preprocessing import *
from ts_missing_values.utility import *

from tqdm import tqdm
from functools import partial

## Preparation

in this notebook we're gonna compare the time-series-filling method with the linear interpolation.

Matrics used: MAE and RMSE
methods: time series filing and linear interpolation

we're gonna see evey combination and make a comparison.

note: every time you find theese are acronyms:
- smv stands for sporadic missing values
- gap is clustered missing values
- tsf is time seris filling
- li is linear interpolation

## Preparation

In this notebook we compare gap-filling methods.

**Methods:**
- TSF — Time Series Filling
- LI — Linear Interpolation

**Missing value types:**
- SMV — Sporadic Missing Values
- GAP — Clustered Missing Values

**Metrics:** MAE, RMSE

All four combinations (TSF/LI × SMV/GAP) are evaluated and compared side by side.

In [ ]:
sensors = pd.read_csv('./data/synthetic_traffic.csv')
sensors.timestamp = pd.to_datetime(sensors.timestamp)
sensors = sensors.set_index('timestamp')

Wrap each sensor into a Darts TimeSeries, plot it for inspection, and store series and names into a dict and list for future use.

In [ ]:
sensors_dict = {}

fig, ax = plt.subplots(10,3, figsize=(15,10))

for i in range(10):
    for j in range(3):
        name = sensors.columns[j+i*3]
        ts_original = TimeSeries(times=sensors.index, values=sensors[name].values)
        ts_original.plot(ax=ax[i,j])
        ax[i,j].get_legend().remove()
        sensors_dict[name] = ts_original
        
sensors_list = list(sensors_dict.values())
sensors_names = list(sensors_dict.keys())

## Validation functions

In [ ]:
def gap_validation(series_list, filling_function, n_cycles_per_series=10):
    all_gap_MAE = []
    all_gap_RMSE = []

    for series_idx, series in tqdm(enumerate(series_list)):
        mae_results = []
        rmse_results = []

        candidates = series_list.copy()
        candidates.remove(series)
        
        for _ in range(n_cycles_per_series):
            series_with_gap, artificial_gap = create_artificial_gap(series, return_removed_slice=True)
            
            if filling_function.__name__ == 'gap_filling':
                # hgd = extract_high_gap_density(series_with_gap)
                _, hgd = gap_transform(series_with_gap, return_density_series=True) 
                filled_series = gap_filling(series=series_with_gap, high_gap_density_series=hgd, candidates=candidates)
            else:
                filled_series = filling_function(series=series_with_gap)

            original_gap = series.slice_intersect(artificial_gap)
            filled_gap = filled_series.slice_intersect(artificial_gap)
            
            if is_series_empty(filled_gap) or is_series_empty(original_gap):
                continue

            mae_results.append(mae(original_gap, filled_gap))
            rmse_results.append(rmse(original_gap, filled_gap))

        all_gap_MAE.append(mae_results)
        all_gap_RMSE.append(rmse_results)
    
    return all_gap_MAE, all_gap_RMSE

In [ ]:
def sporadic_validation(series_list, filling_function, n_cycles_per_series=10, n_sporadic_mv=10):
    all_sporadic_MAE = []
    all_sporadic_RMSE = []

    for series_idx, series in tqdm(enumerate(series_list)):
        mae_results = []
        rmse_results = []
        for _ in range(n_cycles_per_series):
            series_with_sporadic_mv, removed_values_df = create_artificial_sporadic_missing_values(series, number_of_artificial_gaps=n_sporadic_mv)
            removed_dates = removed_values_df.index
            
            if filling_function.__name__ == 'sporadic_filling':
                _, hgd = gap_transform(series_with_sporadic_mv, return_density_series=True) 
                filled_series = sporadic_filling(series=series_with_sporadic_mv, high_gap_density_series=hgd)
            else:
                filled_series = filling_function(series=series_with_sporadic_mv)

            original_values = series.univariate_values()[filled_series.time_index.isin(removed_dates)]
            filled_values = filled_series.univariate_values()[filled_series.time_index.isin(removed_dates)]
            
            filling_mae = np.nanmean(abs(original_values-filled_values))
            filling_rmse = np.sqrt(np.nanmean((original_values - filled_values)**2))

            rmse_results.append(filling_rmse)
            mae_results.append(filling_mae)

        all_sporadic_MAE.append(mae_results)
        all_sporadic_RMSE.append(rmse_results)
        
    return all_sporadic_MAE, all_sporadic_RMSE

In [ ]:
def print_metrics(name, metrics):
    metric_for_each_series = np.nanmean(np.array(metrics), axis=1)
    
    print(f'matrics for {name}')
    print('====================')
    print(f'min {np.nanmin(metric_for_each_series):.2f}')
    print(f'25Q {np.quantile(metric_for_each_series, 0.25):.2f}')
    print(f'mean {np.mean(metric_for_each_series):.2f}')
    print(f'median {np.median(metric_for_each_series):.2f}')
    print(f'75Q {np.quantile(metric_for_each_series, 0.75):.2f}')
    print(f'max {np.nanmax(metric_for_each_series):.2f}')

In [ ]:
def compute_metrics(name, metrics):
    s = np.nanmean(np.array(metrics), axis=1)
    return pd.Series({
        'min':    np.nanmin(s),
        'q25':    np.nanquantile(s, 0.25),
        'mean':   np.nanmean(s),
        'median': np.nanmedian(s),
        'q75':    np.nanquantile(s, 0.75),
        'max':    np.nanmax(s),
    }, name=name)

## Linear interpolation validation

### Gaps

In [ ]:
linear_gap_MAE, linear_gap_RMSE = gap_validation(
    series_list=sensors_list, 
    filling_function=fill_missing_values,
    n_cycles_per_series=50
    )

In [ ]:
li_gap_MAE_stats = compute_metrics('li_gap_MAE', linear_gap_MAE)
li_gap_MAE_stats

In [ ]:
li_gap_RMSE_stats = compute_metrics('li_gap_RMSE', linear_gap_RMSE)
li_gap_RMSE_stats

### Sporadic missing values

In [ ]:
linear_smv_MAE, linear_smv_RMSE = sporadic_validation(
    series_list=sensors_list, 
    filling_function=fill_missing_values,
    n_cycles_per_series=50,
    n_sporadic_mv=25
    )

In [ ]:
li_smv_MAE_stats = compute_metrics('li_smv_MAE', linear_smv_MAE)
li_smv_MAE_stats

In [ ]:
li_smv_RMSE_stats = compute_metrics('li_smv_RMSE', linear_smv_RMSE)
li_smv_RMSE_stats

## time_series_filling validation

## Gaps

In [ ]:
tsf_gap_MAE, tsf_gap_RMSE = gap_validation(
    series_list=sensors_list, 
    filling_function=gap_filling,
    n_cycles_per_series=50
    )

In [ ]:
tsf_gap_MAE_stats = compute_metrics('tsf_gap_MAE', tsf_gap_MAE)
tsf_gap_MAE_stats

In [ ]:
tsf_gap_RMSE_stats = compute_metrics('tsf_gap_RMSE', tsf_gap_RMSE)
tsf_gap_RMSE_stats

### Sporadic missing values

In [ ]:
tsf_sporadic_MAE, tsf_sporadic_RMSE = sporadic_validation(
    series_list=sensors_list, 
    filling_function=sporadic_filling,
    n_cycles_per_series=50,
    n_sporadic_mv=25
    )

In [ ]:
tsf_smv_MAE_stats = compute_metrics('tsf_smv_MAE', tsf_sporadic_MAE)
tsf_smv_MAE_stats

In [ ]:
tsf_smv_RMSE_stats = compute_metrics('tsf_smv_RMSE', tsf_sporadic_RMSE)
tsf_smv_RMSE_stats

## Comparison

In [ ]:
mae_comparison = pd.concat([
    li_gap_MAE_stats, tsf_gap_MAE_stats,
    li_smv_MAE_stats, tsf_smv_MAE_stats
    ], axis=1)

mae_comparison

In [ ]:
rmse_comparison = pd.concat([
    li_gap_RMSE_stats, tsf_gap_RMSE_stats,
    li_smv_RMSE_stats, tsf_smv_RMSE_stats
    ], axis=1)

rmse_comparison